In [1]:

# ================================================================
# NEUROGAP AI
# Neuroscience Research + Public Awareness Intelligence Platform
# Single-cell Google Colab / Jupyter demonstration
#
# REAL DATA: PubMed metadata and abstracts fetched from NCBI.
# SYNTHETIC DATA: Awareness responses generated only for demo.
# NOT A MEDICAL DIAGNOSTIC OR CLINICAL PREDICTION SYSTEM.
# ================================================================

import sys, subprocess, importlib.util, warnings, re, time, json
warnings.filterwarnings("ignore")

# 1. Install any missing dependencies
packages = {
    "requests": "requests",
    "pandas": "pandas",
    "numpy": "numpy",
    "plotly": "plotly",
    "sklearn": "scikit-learn",
    "ipywidgets": "ipywidgets"
}
for module, package in packages.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

import requests
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, HTML, Markdown, clear_output
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from datetime import datetime
from urllib.parse import quote_plus
import ipywidgets as widgets

# 2. Configuration
PROJECT_NAME = "NEUROGAP AI"
MAX_PAPERS_PER_TOPIC = 40
NCBI_BASE = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
HEADERS = {"User-Agent": "NeuroGapAI-StudentResearch/1.0"}
np.random.seed(42)

TOPICS = {
    "Alzheimer's disease": {
        "query": '"Alzheimer Disease"[Title/Abstract]',
        "keywords": ["alzheimer", "amyloid", "tau", "memory", "dementia"]
    },
    "Parkinson's disease": {
        "query": '"Parkinson Disease"[Title/Abstract]',
        "keywords": ["parkinson", "dopamine", "motor", "alpha-synuclein", "tremor"]
    },
    "Early detection & biomarkers": {
        "query": '(neurodegeneration OR Alzheimer OR Parkinson) AND (biomarker OR "early detection")',
        "keywords": ["biomarker", "early detection", "diagnosis", "screening", "blood test"]
    },
    "Risk factors & prevention": {
        "query": '(neurodegeneration OR Alzheimer OR Parkinson) AND ("risk factor" OR prevention OR lifestyle)',
        "keywords": ["risk factor", "prevention", "exercise", "sleep", "lifestyle"]
    },
    "Caregiver & public awareness": {
        "query": '(dementia OR Alzheimer OR Parkinson) AND (awareness OR knowledge OR caregiver)',
        "keywords": ["awareness", "knowledge", "caregiver", "stigma", "education"]
    },
    "AI & computational neuroscience": {
        "query": '(neurodegeneration OR Alzheimer OR Parkinson) AND (artificial intelligence OR machine learning OR computational)',
        "keywords": ["artificial intelligence", "machine learning", "deep learning", "computational", "model"]
    }
}

# 3. Fetch actual PubMed articles via NCBI E-utilities
def pubmed_search(term, retmax=40):
    """Return real PubMed metadata and abstracts, or an empty list."""
    try:
        params = {
            "db": "pubmed", "term": term, "retmax": retmax,
            "retmode": "json", "sort": "relevance",
            "tool": "NeuroGapAIStudentProject",
            "email": "student@example.com"
        }
        response = requests.get(
            NCBI_BASE + "esearch.fcgi", params=params,
            headers=HEADERS, timeout=25
        )
        response.raise_for_status()
        ids = response.json().get("esearchresult", {}).get("idlist", [])
        if not ids:
            return []

        # Respect NCBI's request rate guidance.
        time.sleep(0.4)
        params2 = {
            "db": "pubmed", "id": ",".join(ids),
            "retmode": "xml", "tool": "NeuroGapAIStudentProject",
            "email": "student@example.com"
        }
        r = requests.get(
            NCBI_BASE + "efetch.fcgi", params=params2,
            headers=HEADERS, timeout=40
        )
        r.raise_for_status()

        import xml.etree.ElementTree as ET
        root = ET.fromstring(r.text)
        records = []
        for article in root.findall(".//PubmedArticle"):
            pmid = article.findtext(".//PMID", default="")
            title = " ".join(
                "".join(article.find(".//ArticleTitle").itertext()).split()
            ) if article.find(".//ArticleTitle") is not None else "Title unavailable"

            abstract_parts = []
            for node in article.findall(".//Abstract/AbstractText"):
                label = node.attrib.get("Label", "")
                value = " ".join("".join(node.itertext()).split())
                abstract_parts.append((label + ": " if label else "") + value)
            abstract = " ".join(abstract_parts)

            journal = article.findtext(".//Journal/Title", default="Unknown journal")
            year = article.findtext(".//PubDate/Year", default="")
            if not year:
                medline_date = article.findtext(".//PubDate/MedlineDate", default="")
                match = re.search(r"\b(19|20)\d{2}\b", medline_date)
                year = match.group(0) if match else "Unknown"

            records.append({
                "pmid": pmid, "title": title,
                "abstract": abstract, "journal": journal,
                "year": str(year),
                "url": f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/"
            })
        return records
    except Exception as e:
        print(f"PubMed retrieval issue: {str(e)[:180]}")
        return []

# 4. Download papers and tag their research topics
papers = []
topic_fetch_status = {}
seen_pmids = set()

print("Fetching research from PubMed. This can take a minute...\n")
for topic, config in TOPICS.items():
    records = pubmed_search(config["query"], MAX_PAPERS_PER_TOPIC)
    new_count = 0
    for record in records:
        if record["pmid"] and record["pmid"] not in seen_pmids:
            seen_pmids.add(record["pmid"])
            record["search_topic"] = topic
            record["text"] = (record["title"] + " " + record["abstract"]).strip()
            papers.append(record)
            new_count += 1
    topic_fetch_status[topic] = {
        "returned": len(records), "unique_added": new_count
    }
    print(f"  {topic}: {len(records)} records retrieved, {new_count} unique added")
    time.sleep(0.4)

# Do not fabricate papers if API access fails.
papers_df = pd.DataFrame(papers)
if papers_df.empty:
    display(HTML("""
    <div style='padding:18px;border:1px solid #d97706;border-radius:12px'>
    <h3>PubMed data could not be retrieved</h3>
    Check your internet connection and rerun this cell.
    No fake research papers have been substituted for live results.
    </div>
    """))
    raise RuntimeError("No live PubMed records retrieved. Retry when internet is available.")

# 5. NLP-based theme classification
# A paper can match more than one theme.
def tag_themes(text):
    text = str(text).lower()
    found = []
    for topic, config in TOPICS.items():
        if any(k.lower() in text for k in config["keywords"]):
            found.append(topic)
    return found if found else ["Other / unclassified"]

papers_df["themes"] = papers_df["text"].apply(tag_themes)
papers_df["abstract_length"] = papers_df["abstract"].str.len()
papers_df["has_abstract"] = papers_df["abstract"].str.strip().ne("")
papers_df["year_numeric"] = pd.to_numeric(papers_df["year"], errors="coerce")
papers_df["publication_type"] = "PubMed record; article type not classified"

# TF-IDF is used for transparent text similarity, not clinical inference.
texts = papers_df["text"].fillna("").tolist()
try:
    vectorizer = TfidfVectorizer(
        stop_words="english", max_features=2500,
        ngram_range=(1, 2), min_df=1
    )
    tfidf_matrix = vectorizer.fit_transform(texts)
except Exception:
    vectorizer, tfidf_matrix = None, None

# 6. Create a clearly labeled synthetic awareness survey.
# This is demo data, NOT real participants or actual public opinion.
N_SYNTHETIC = 240
rng = np.random.default_rng(42)
survey_topics = list(TOPICS.keys())
synthetic_profiles = {
    "Alzheimer's disease": 0.78,
    "Parkinson's disease": 0.61,
    "Early detection & biomarkers": 0.39,
    "Risk factors & prevention": 0.53,
    "Caregiver & public awareness": 0.43,
    "AI & computational neuroscience": 0.24
}
survey_rows = []
for i in range(N_SYNTHETIC):
    age_group = rng.choice(["18–24", "25–34", "35–44", "45–59", "60+"],
                           p=[.30, .27, .19, .15, .09])
    education = rng.choice(
        ["High school", "Undergraduate", "Postgraduate", "Other"],
        p=[.22, .40, .27, .11]
    )
    for topic in survey_topics:
        base = synthetic_profiles[topic]
        knowledge = int(rng.binomial(1, base))
        confidence = int(rng.integers(1, 6))
        survey_rows.append({
            "respondent_id": f"SYN-{i+1:03d}",
            "age_group": age_group,
            "education": education,
            "topic": topic,
            "self_reported_awareness": knowledge,
            "confidence_1_to_5": confidence,
            "data_source": "SYNTHETIC DEMO — not real survey responses"
        })
survey_df = pd.DataFrame(survey_rows)

awareness = (
    survey_df.groupby("topic", as_index=False)
    .agg(awareness_rate=("self_reported_awareness", "mean"),
         mean_confidence=("confidence_1_to_5", "mean"),
         demo_responses=("respondent_id", "nunique"))
)
awareness["awareness_pct"] = awareness["awareness_rate"] * 100
awareness["mean_confidence"] = awareness["mean_confidence"].round(2)

# 7. Count research papers by topic.
research_rows = []
for topic in survey_topics:
    count = int(papers_df["themes"].apply(lambda ts: topic in ts).sum())
    research_rows.append({"topic": topic, "paper_count": count})
research = pd.DataFrame(research_rows)

# 8. Compute a project-defined relative research-coverage score.
# This measures representation in this retrieved sample, NOT the total field.
max_count = max(research["paper_count"].max(), 1)
research["coverage_pct_of_max"] = research["paper_count"] / max_count * 100
gap = research.merge(awareness, on="topic", how="left")
gap["gap_score"] = (
    0.5 * (100 - gap["coverage_pct_of_max"]) +
    0.5 * (100 - gap["awareness_pct"])
).round(1)
gap["interpretation"] = np.select(
    [
        (gap["coverage_pct_of_max"] < 50) & (gap["awareness_pct"] < 50),
        (gap["coverage_pct_of_max"] >= 50) & (gap["awareness_pct"] < 50),
        (gap["coverage_pct_of_max"] < 50) & (gap["awareness_pct"] >= 50)
    ],
    [
        "Low coverage and low synthetic awareness",
        "Research visible in sample; synthetic awareness lower",
        "Synthetic awareness higher; research coverage lower"
    ],
    default="Higher relative coverage and synthetic awareness"
)

# 9. Professional dashboard header
display(HTML("""
<div style="
background:linear-gradient(120deg,#0b1220,#162d50);
color:white;padding:28px;border-radius:18px;margin-bottom:15px">
<div style="font-size:13px;letter-spacing:3px;color:#7dd3fc">
RESEARCH INTELLIGENCE • BIOINFORMATICS • NLP</div>
<h1 style="font-size:34px;margin:8px 0">🧠 NEUROGAP AI</h1>
<div style="font-size:17px">Neurodegeneration Research & Awareness Explorer</div>
<p style="color:#cbd5e1;margin-bottom:0">
Discover research themes, inspect source papers, and explore potential
knowledge gaps using transparent data analysis.</p>
</div>
"""))

# 10. KPI cards
total_papers = len(papers_df)
abstracts_available = int(papers_df["has_abstract"].sum())
unique_years = int(papers_df["year_numeric"].dropna().nunique())
kpis = [
    ("LIVE PUBMED RECORDS", f"{total_papers:,}"),
    ("RECORDS WITH ABSTRACTS", f"{abstracts_available:,}"),
    ("RESEARCH THEMES", f"{len(TOPICS)}"),
    ("PUBLICATION YEARS FOUND", f"{unique_years}")
]
cards = "".join([
    f"""<div style='flex:1;min-width:150px;background:#f1f5f9;
    padding:17px;border-radius:12px;border:1px solid #dbeafe'>
    <div style='font-size:11px;color:#475569'>{label}</div>
    <div style='font-size:27px;font-weight:700;color:#0f172a'>{value}</div>
    </div>"""
    for label, value in kpis
])
display(HTML(f"<div style='display:flex;flex-wrap:wrap;gap:12px'>{cards}</div>"))

display(Markdown(
    "**Data integrity:** PubMed records are live retrieval results. "
    "The awareness figures below are synthetic demonstration data only. "
    "The gap score is an exploratory project metric, not a validated "
    "scientific or clinical measure."
))

# 11. Visualization A: research coverage
fig1 = px.bar(
    research.sort_values("paper_count", ascending=True),
    x="paper_count", y="topic", orientation="h",
    text="paper_count", color="paper_count",
    color_continuous_scale="Blues",
    title="Research Coverage by Topic",
    labels={"paper_count": "Matching papers in retrieved sample", "topic": ""}
)
fig1.update_layout(
    template="plotly_white", height=430,
    coloraxis_showscale=False,
    xaxis_title="Number of retrieved papers matching keyword rules",
    yaxis_title="", margin=dict(l=10,r=20,t=65,b=30)
)
fig1.update_traces(textposition="outside")
fig1.show()

# 12. Visualization B: year distribution
year_df = papers_df[papers_df["year_numeric"].between(1900, 2100)].copy()
if not year_df.empty:
    year_counts = (
        year_df.groupby(year_df["year_numeric"].astype(int))
        .size().reset_index(name="papers")
        .rename(columns={"year_numeric": "year"})
    )
    fig2 = px.line(
        year_counts, x="year", y="papers", markers=True,
        title="Publication-Year Distribution of Retrieved Records",
        labels={"year": "Publication year", "papers": "Retrieved papers"}
    )
    fig2.update_layout(template="plotly_white", height=350)
    fig2.show()
else:
    print("No usable publication years were returned by PubMed.")

# 13. Visualization C: synthetic awareness levels
fig3 = px.bar(
    awareness.sort_values("awareness_pct"),
    x="awareness_pct", y="topic", orientation="h",
    text=awareness.sort_values("awareness_pct")["awareness_pct"].map(lambda x: f"{x:.1f}%"),
    color="awareness_pct", color_continuous_scale="Teal",
    title="Synthetic Awareness Levels — Demonstration Only",
    labels={"awareness_pct": "Synthetic awareness rate (%)", "topic": ""}
)
fig3.update_layout(
    template="plotly_white", height=430, coloraxis_showscale=False,
    xaxis=dict(range=[0,100], title="Synthetic awareness rate (%)"),
    yaxis_title=""
)
fig3.show()

# 14. Visualization D: research coverage versus awareness
fig4 = px.scatter(
    gap, x="coverage_pct_of_max", y="awareness_pct",
    size="paper_count", color="topic", hover_name="topic",
    hover_data={
        "paper_count": True,
        "coverage_pct_of_max": ":.1f",
        "awareness_pct": ":.1f",
        "gap_score": True
    },
    title="Research Coverage vs Synthetic Awareness",
    labels={
        "coverage_pct_of_max": "Relative coverage in retrieved sample (%)",
        "awareness_pct": "Synthetic awareness rate (%)",
        "paper_count": "Matching papers"
    }
)
fig4.add_vline(x=50, line_dash="dash", line_color="gray")
fig4.add_hline(y=50, line_dash="dash", line_color="gray")
fig4.update_layout(template="plotly_white", height=500)
fig4.show()

# 15. Visualization E: gap matrix heatmap
heat_data = gap.set_index("topic")[["coverage_pct_of_max", "awareness_pct"]]
heat_data.columns = ["Relative research coverage (%)", "Synthetic awareness (%)"]
fig5 = px.imshow(
    heat_data, text_auto=".1f", aspect="auto",
    color_continuous_scale="RdYlGn",
    zmin=0, zmax=100,
    title="NeuroGap Matrix — Coverage and Awareness"
)
fig5.update_layout(template="plotly_white", height=400)
fig5.show()

# 16. Gap ranking table
display(Markdown("## Potential gap priorities"))
display(Markdown(
    "Higher scores indicate lower values on the two project-defined dimensions. "
    "They do not prove a true research gap; validate through systematic review "
    "and real, representative survey data."
))
gap_display = gap[[
    "topic", "paper_count", "coverage_pct_of_max",
    "awareness_pct", "gap_score", "interpretation"
]].sort_values("gap_score", ascending=False).copy()
gap_display.columns = [
    "Topic", "Matching papers", "Relative coverage (%)",
    "Synthetic awareness (%)", "Exploratory gap score", "Interpretation"
]
display(gap_display.round(1).style.format({
    "Relative coverage (%)": "{:.1f}",
    "Synthetic awareness (%)": "{:.1f}",
    "Exploratory gap score": "{:.1f}"
}).background_gradient(subset=["Exploratory gap score"], cmap="YlOrRd"))

# 17. Research paper explorer
display(Markdown("## Evidence Explorer"))
display(Markdown(
    "Search the retrieved titles and abstracts. Results link directly to PubMed. "
    "This tool retrieves source text; it does not independently verify medical claims."
))
search_box = widgets.Text(
    value="", placeholder="e.g. tau, caregiver, biomarker, machine learning",
    description="Search:", layout=widgets.Layout(width="75%")
)
topic_filter = widgets.Dropdown(
    options=["All topics"] + survey_topics + ["Other / unclassified"],
    value="All topics", description="Topic:",
    layout=widgets.Layout(width="75%")
)
max_results = widgets.IntSlider(
    value=10, min=5, max=30, step=5, description="Results:",
    continuous_update=False
)
explorer_output = widgets.Output()

def update_explorer(change=None):
    with explorer_output:
        clear_output(wait=True)
        df = papers_df.copy()
        selected = topic_filter.value
        if selected != "All topics":
            df = df[df["themes"].apply(lambda x: selected in x)]
        query = search_box.value.strip().lower()
        if query:
            df = df[df["text"].str.lower().str.contains(re.escape(query), na=False)]
        if df.empty:
            print("No matching records. Try another keyword or topic.")
            return
        for _, row in df.head(max_results.value).iterrows():
            title = str(row["title"])
            abstract = str(row["abstract"])
            if not abstract:
                abstract = "No abstract was supplied in this PubMed record."
            display(HTML(f"""
            <div style='border:1px solid #cbd5e1;border-radius:10px;
            padding:14px;margin:9px 0'>
            <div style='font-size:11px;color:#64748b'>
            PMID {row['pmid']} · {row['year']} · {row['journal']}</div>
            <div style='font-size:16px;font-weight:700;margin:5px 0'>
            <a href='{row['url']}' target='_blank'>{title}</a></div>
            <div style='font-size:12px;color:#0f766e'>
            {', '.join(row['themes'])}</div>
            <div style='margin-top:7px;font-size:13px;line-height:1.5'>
            {abstract[:1200]}{'…' if len(abstract)>1200 else ''}
            </div></div>"""))
        print(f"Showing {min(len(df), max_results.value)} of {len(df)} matching records.")

for control in [search_box, topic_filter, max_results]:
    control.observe(update_explorer, names="value")
display(widgets.VBox([search_box, topic_filter, max_results, explorer_output]))
update_explorer()

# 18. Simple evidence-linked semantic search (TF-IDF)
display(Markdown("## Research Question Explorer"))
display(Markdown(
    "Enter a question to retrieve textually similar records. Similarity is "
    "based on word/phrase overlap, not medical reasoning or proof of causality."
))
question_box = widgets.Textarea(
    value="What research discusses early detection biomarkers in Alzheimer's disease?",
    description="Question:",
    layout=widgets.Layout(width="95%", height="75px")
)
question_button = widgets.Button(
    description="Find relevant papers",
    button_style="primary",
    icon="search"
)
question_output = widgets.Output()

def answer_question(_):
    with question_output:
        clear_output(wait=True)
        question = question_box.value.strip()
        if not question or vectorizer is None or tfidf_matrix is None:
            print("Enter a question and ensure text indexing completed.")
            return
        q_vector = vectorizer.transform([question])
        scores = cosine_similarity(q_vector, tfidf_matrix).ravel()
        best = np.argsort(scores)[::-1][:5]
        display(Markdown(
            "**Retrieved source records — review the linked papers before drawing conclusions.**"
        ))
        for idx in best:
            row = papers_df.iloc[idx]
            display(HTML(f"""
            <div style='padding:12px;border-left:4px solid #0d9488;
            margin:10px 0;background:#f8fafc'>
            <b><a href='{row['url']}' target='_blank'>{row['title']}</a></b><br>
            <small>Similarity score: {scores[idx]:.3f} · PMID {row['pmid']} · {row['year']}</small><br>
            {str(row['abstract'])[:650]}{'…' if len(str(row['abstract']))>650 else ''}
            </div>"""))

question_button.on_click(answer_question)
display(widgets.VBox([question_box, question_button, question_output]))
answer_question(None)

# 19. Data quality checks
display(Markdown("## Data Quality & Methodology"))
quality = pd.DataFrame([
    ["PubMed records retrieved", len(papers_df), "Live source records"],
    ["Unique PMIDs", papers_df["pmid"].nunique(), "Should match retrieved unique records"],
    ["Records with abstracts", int(papers_df["has_abstract"].sum()), "Abstract availability varies"],
    ["Records without abstracts", int((~papers_df["has_abstract"]).sum()), "May still have title and metadata"],
    ["Duplicate PMIDs", int(papers_df["pmid"].duplicated().sum()), "Expected 0 after deduplication"],
    ["Synthetic survey respondents", N_SYNTHETIC, "Artificial demonstration data only"],
    ["Synthetic topic-level response rows", len(survey_df), "Not real human participants"],
])
quality.columns = ["Quality check", "Result", "Explanation"]
display(quality)

display(Markdown("""
### Research limitations
1. PubMed search results depend on the query, ranking, and retrieval limit.
2. Keyword matching can miss relevant papers and incorrectly classify unrelated text.
3. A larger retrieved paper count does not establish research quality or importance.
4. Synthetic awareness data must never be reported as a real survey finding.
5. A genuine awareness study needs an approved survey plan, informed consent,
   privacy safeguards, and appropriate sampling.
6. This platform is for research exploration, not diagnosis, treatment advice,
   or clinical decision-making.
"""))

# 20. Export reproducible outputs
papers_export = papers_df[[
    "pmid", "title", "journal", "year", "abstract", "url", "themes"
]].copy()
papers_export["themes"] = papers_export["themes"].apply(lambda x: "; ".join(x))

papers_export.to_csv("neurogap_pubmed_records.csv", index=False)
gap.to_csv("neurogap_exploratory_gap_analysis.csv", index=False)
survey_df.to_csv("neurogap_SYNTHETIC_awareness_demo.csv", index=False)

with open("neurogap_methodology.json", "w", encoding="utf-8") as f:
    json.dump({
        "project": PROJECT_NAME,
        "generated_at": datetime.now().isoformat(timespec="seconds"),
        "source": "NCBI PubMed E-utilities",
        "retrieved_unique_records": int(len(papers_df)),
        "survey_data": "SYNTHETIC DEMO ONLY — not real survey participants",
        "topic_classification": "Transparent keyword matching",
        "gap_score": "Mean of (100 - relative research coverage) and (100 - synthetic awareness)",
        "clinical_use": False,
        "limitations": [
            "Search query and retrieval limit affect results",
            "Keyword classification is imperfect",
            "Gap score is exploratory and unvalidated",
            "Synthetic survey cannot support real-world awareness claims"
        ]
    }, f, indent=2)

print("\n" + "="*65)
print("NEUROGAP AI RUN COMPLETED")
print("="*65)
print(f"Live PubMed records: {len(papers_df)}")
print("Exported files:")
print("  1. neurogap_pubmed_records.csv")
print("  2. neurogap_exploratory_gap_analysis.csv")
print("  3. neurogap_SYNTHETIC_awareness_demo.csv")
print("  4. neurogap_methodology.json")
print("\nIn Google Colab, open the Files panel to download these CSV/JSON files.")
print("Next step: replace the synthetic survey with real, ethically collected responses.")


Fetching research from PubMed. This can take a minute...

  Alzheimer's disease: 39 records retrieved, 39 unique added
  Parkinson's disease: 38 records retrieved, 38 unique added
  Early detection & biomarkers: 40 records retrieved, 40 unique added
  Risk factors & prevention: 40 records retrieved, 37 unique added
  Caregiver & public awareness: 40 records retrieved, 37 unique added
  AI & computational neuroscience: 40 records retrieved, 39 unique added


**Data integrity:** PubMed records are live retrieval results. The awareness figures below are synthetic demonstration data only. The gap score is an exploratory project metric, not a validated scientific or clinical measure.

## Potential gap priorities

Higher scores indicate lower values on the two project-defined dimensions. They do not prove a true research gap; validate through systematic review and real, representative survey data.

,Topic,Matching papers,Relative coverage (%),Synthetic awareness (%),Exploratory gap score,Interpretation
5,AI & computational neuroscience,67,36.8,24.2,69.5,Low coverage and low synthetic awareness
4,Caregiver & public awareness,46,25.3,42.9,65.9,Low coverage and low synthetic awareness
3,Risk factors & prevention,43,23.6,56.2,60.1,Synthetic awareness higher; research coverage lower
2,Early detection & biomarkers,77,42.3,44.2,56.8,Low coverage and low synthetic awareness
1,Parkinson's disease,182,100.0,65.4,17.3,Higher relative coverage and synthetic awareness
0,Alzheimer's disease,163,89.6,77.1,16.7,Higher relative coverage and synthetic awareness


...

Enter a question to retrieve textually similar records. Similarity is based on word/phrase overlap, not medical reasoning or proof of causality.

## Data Quality & Methodology

,Quality check,Result,Explanation
0,PubMed records retrieved,230,Live source records
1,Unique PMIDs,230,Should match retrieved unique records
2,Records with abstracts,209,Abstract availability varies
3,Records without abstracts,21,May still have title and metadata
4,Duplicate PMIDs,0,Expected 0 after deduplication
5,Synthetic survey respondents,240,Artificial demonstration data only
6,Synthetic topic-level response rows,1440,Not real human participants



### Research limitations
1. PubMed search results depend on the query, ranking, and retrieval limit.
2. Keyword matching can miss relevant papers and incorrectly classify unrelated text.
3. A larger retrieved paper count does not establish research quality or importance.
4. Synthetic awareness data must never be reported as a real survey finding.
5. A genuine awareness study needs an approved survey plan, informed consent,
   privacy safeguards, and appropriate sampling.
6. This platform is for research exploration, not diagnosis, treatment advice,
   or clinical decision-making.



NEUROGAP AI RUN COMPLETED
Live PubMed records: 230
Exported files:
  1. neurogap_pubmed_records.csv
  2. neurogap_exploratory_gap_analysis.csv
  3. neurogap_SYNTHETIC_awareness_demo.csv
  4. neurogap_methodology.json

In Google Colab, open the Files panel to download these CSV/JSON files.
Next step: replace the synthetic survey with real, ethically collected responses.
